Text coherence new method using word2vec sentence vectors and most likely n-grams (10.1109/ICSPIS.2017.8311598)

In [10]:
import nltk
import numpy as np
from gensim.models import Word2Vec
import gensim.downloader as api
from gensim.models import KeyedVectors
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
# Load or download required NLTK data
nltk.download('punkt')
model = KeyedVectors.load_word2vec_format(api.load("word2vec-google-news-300", return_path=True), binary=True)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [12]:
# Define helper functions

def preprocess_text(text):
    """
    Preprocess text by tokenizing it into sentences and words.
    """
    sentences = sent_tokenize(text)  # Split text into sentences
    tokenized_sentences = [word_tokenize(sent) for sent in sentences]  # Tokenize words in each sentence
    return tokenized_sentences

def sentence_to_matrix(sentence, model):
    """
    Convert a sentence to a matrix using Word2Vec vectors.
    """
    sentence_matrix = []
    for word in sentence:
        if word in model:
            sentence_matrix.append(model[word])
        else:
            sentence_matrix.append(np.zeros(model.vector_size))  # Fallback for unknown words
    return np.array(sentence_matrix)

def normalize_sentence_matrix(sentence_matrix, n_gram_size=5):
    """
    Normalize the sentence matrix by averaging word vectors using n-grams.
    """
    n = min(len(sentence_matrix), n_gram_size)
    avg_n_gram_matrix = []
    for i in range(0, len(sentence_matrix) - n + 1):
        n_gram = sentence_matrix[i:i + n]
        avg_n_gram = np.mean(n_gram, axis=0)
        avg_n_gram_matrix.append(avg_n_gram)
    return np.array(avg_n_gram_matrix)

def calculate_coherence(sentences, model):
    """
    Calculate the coherence of a text by computing the similarities between adjacent sentences.
    """
    coherence_scores = []
    for i in range(len(sentences) - 1):
        matrix1 = normalize_sentence_matrix(sentence_to_matrix(sentences[i], model))
        matrix2 = normalize_sentence_matrix(sentence_to_matrix(sentences[i + 1], model))
        
        # Use cosine similarity between normalized sentence matrices to assess coherence
        similarity = cosine_similarity(matrix1.mean(axis=0).reshape(1, -1), matrix2.mean(axis=0).reshape(1, -1))
        coherence_scores.append(similarity[0][0])
    
    return np.mean(coherence_scores)

def evaluate_text_coherence(text, model):
    """
    Main function to evaluate the coherence of a given text.
    """
    sentences = preprocess_text(text)
    coherence = calculate_coherence(sentences, model)
    return coherence

In [13]:
# Example usage
coherent_text = """
Bees are remarkable insects that play a crucial role in our ecosystem, primarily through pollination. Transferring pollen between flowers, they help plants reproduce, supporting biodiversity and the growth of crops that humans and animals rely on for food. Honeybees, in particular, are known for producing honey and beeswax, both valuable human resources. There are over 20,000 species of bees, ranging from solitary bees to highly social species like the honeybee.
"""
coherence_score = evaluate_text_coherence(coherent_text, model)
print(f"Coherence Score: {coherence_score}")

AttributeError: 'KeyedVectors' object has no attribute 'wv'

In [ ]:
# Test with a less coherent text
incoherent_text = """
Bees are insects that have an impact, in our environment by mainly aiding in pollination activities which support plant reproduction and the diversity of crops crucial for our food sources and animal sustenance alike. Honeybees stand out for their production of honey and beeswax that hold value for humans as resources within our society. The bee population comprises than 20 thousand species that include bees as well as highly sociable species such, as the honeybee.
"""
coherence_score = evaluate_text_coherence(incoherent_text, model)
print(f"Coherence Score: {coherence_score}")